In [ ]:
# =========================================================
# Install packages
# =========================================================

!pip -q install langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers

In [ ]:
# =========================================================
# Imports
# =========================================================

#!pip -q install langchain langchain-community langchain-huggingface langchain-chroma chromadb sentence-transformers

import re

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# =========================================================
# Creating small documents
# =========================================================

documents = [
    Document(
        page_content="""
        Customers can request a refund within 30 days of purchase.
        The item must be unused and returned in its original packaging.
        Refunds are issued to the original payment method.
        """,
        metadata={"doc_id": "doc1", "title": "Refund Policy"}
    ),
    Document(
        page_content="""
        Damaged products must be reported within 7 days of delivery.
        Customers should send a photo of the damaged item.
        After approval, the company will offer a replacement or refund.
        """,
        metadata={"doc_id": "doc2", "title": "Damaged Products"}
    ),
    Document(
        page_content="""
        Standard shipping takes 3 to 5 business days.
        Express shipping takes 1 to 2 business days.
        International shipping may take longer.
        """,
        metadata={"doc_id": "doc3", "title": "Shipping Policy"}
    ),
    Document(
        page_content="""
        Section 12.4 states that either party may terminate the agreement
        with 30 days written notice if the other party violates the contract.
        """,
        metadata={"doc_id": "doc4", "title": "Section 12.4 Termination"}
    ),
    Document(
        page_content="""
        A physician must review the patient record before treatment.
        The doctor should confirm allergies and medication history.
        """,
        metadata={"doc_id": "doc5", "title": "Medical Synonyms Example"}
    )
]

In [ ]:
# =========================================================
# Split documents into chunks
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=180,
    chunk_overlap=40
)

chunks = text_splitter.split_documents(documents)

print(f"Number of original documents: {len(documents)}")
print(f"Number of chunks: {len(chunks)}")

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"{chunk.metadata['doc_id']}_chunk_{i}"

    print("\n" + "-" * 60)
    print("Chunk ID:", chunk.metadata["chunk_id"])
    print("Title:", chunk.metadata["title"])
    print("Text:", chunk.page_content.strip())

In [ ]:
# =========================================================
# Create embeddings
# =========================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

In [ ]:
# =========================================================
# Store chunks in Chroma vector database
# =========================================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="teaching_rag_demo"
)

print("Chunks stored in vector database.")

In [ ]:
# =========================================================
# Create retriever
# =========================================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 3}
)

question = "What is the return deadline for damaged products?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)

for doc in retrieved_docs:
    print("\n" + "-" * 60)
    print("Title:", doc.metadata["title"])
    print("Chunk ID:", doc.metadata["chunk_id"])
    print("Text:", doc.page_content.strip())

In [ ]:
# =========================================================
# Helper functions for RAG
# =========================================================

def format_docs(docs):
    """
    Turns retrieved documents into one context string.
    """
    formatted = []

    for doc in docs:
        title = doc.metadata.get("title", "Unknown Title")
        chunk_id = doc.metadata.get("chunk_id", "Unknown Chunk")

        formatted.append(
            f"Title: {title}\n"
            f"Chunk ID: {chunk_id}\n"
            f"Text: {doc.page_content.strip()}"
        )

    return "\n\n".join(formatted)


def simple_extract_answer(inputs):
    """
    Simple generator.

    This is not a real LLM.
    It selects the sentence from the retrieved context
    with the highest word overlap with the question.
    """

    question = inputs["question"]
    context = inputs["context"]

    sentences = re.split(r"(?<=[.!?])\s+", context)

    question_words = set(re.findall(r"\b\w+\b", question.lower()))

    best_sentence = None
    best_score = 0

    for sentence in sentences:
        sentence_words = set(re.findall(r"\b\w+\b", sentence.lower()))
        score = len(question_words.intersection(sentence_words))

        if score > best_score:
            best_score = score
            best_sentence = sentence

    if best_sentence:
        return best_sentence.strip()

    return "I do not know based on the provided context."

In [ ]:
# =========================================================
# Create a LangChain RAG chain
# =========================================================

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | RunnableLambda(simple_extract_answer)
)

question = "What is the return deadline for damaged products?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

In [ ]:
# =========================================================
# Show answer with sources
# =========================================================
def rag_with_sources(question):
    """
    Runs retrieval and answer generation.
    Returns both the answer and the retrieved sources.
    """

    docs = retriever.invoke(question)
    context = format_docs(docs)

    answer = simple_extract_answer({"question": question, "context": context})

    return {
        "question": question,
        "answer": answer,
        "sources": docs}

result = rag_with_sources("What is the return deadline for damaged products?")

print("Question:")
print(result["question"])

print("\nAnswer:")
print(result["answer"])

print("\nSources:")
for doc in result["sources"]:
    print("\n" + "-" * 60)
    print("Title:", doc.metadata["title"])
    print("Chunk ID:", doc.metadata["chunk_id"])
    print("Text:", doc.page_content.strip())

In [ ]:
# =========================================================
# Try different questions
# =========================================================

questions = [
    "What is the return deadline for damaged products?",
    "How long does standard shipping take?",
    "What does Section 12.4 say about termination?",
    "Who should review the patient record?",
    "What is the refund policy?"
]

for q in questions:
    result = rag_with_sources(q)

    print("\n" + "=" * 80)
    print("Question:", result["question"])
    print("Answer:", result["answer"])

    print("\nTop source:")
    top_source = result["sources"][0]
    print("Title:", top_source.metadata["title"])
    print("Chunk ID:", top_source.metadata["chunk_id"])

In [ ]:
# =========================================================
# Inspect similarity scores
# =========================================================

question = "What does Section 12.4 say about termination?"

docs_and_scores = vector_store.similarity_search_with_score(
    question,
    k=3
)

print("Question:", question)

for doc, score in docs_and_scores:
    print("\n" + "-" * 60)
    print("Score:", score)
    print("Title:", doc.metadata["title"])
    print("Chunk ID:", doc.metadata["chunk_id"])
    print("Text:", doc.page_content.strip())

OpenAI API

In [ ]:
# =========================================================
# Use a real LLM if you have an OpenAI API key
# =========================================================
!pip -q install langchain-openai
import os
from getpass import getpass
from langchain_openai import ChatOpenAI

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template("""
Use the context to answer the question.
If the answer is not in the context, say "I do not know."

Question:
{question}

Context:
{context}
""")

llm_rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the return deadline for damaged products?"
answer = llm_rag_chain.invoke(question)
print(answer)

Gemini API

In [ ]:
# =========================================================
# Use Gemini API as the LLM
# =========================================================

!pip -q install langchain-google-genai

import os
from getpass import getpass

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Google Gemini API key: ")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
Use the context to answer the question.
If the answer is not in the context, say "I do not know."

Question:
{question}

Context:
{context}
""")

llm_rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the return deadline for damaged products?"
answer = llm_rag_chain.invoke(question)

print(answer)

In [ ]:
question = "What is the return deadline for damaged products?"

docs = retriever.invoke(question)
context = format_docs(docs)

print("Retrieved Context:")
print(context)

Groq API

In [ ]:
# =========================================================
# Use Groq API as the LLM
# =========================================================

!pip -q install langchain-groq

import os
from getpass import getpass

from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

prompt = ChatPromptTemplate.from_template("""
Use the context to answer the question.
If the answer is not in the context, say "I do not know."

Question:
{question}

Context:
{context}
""")

llm_rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

question = "What is the return deadline for damaged products?"
answer = llm_rag_chain.invoke(question)

print(answer)

#REAL PROJECT

In [ ]:
# =========================================================
# Install packages
# =========================================================

!pip -q install langchain langchain-community langchain-huggingface langchain-chroma langchain-groq chromadb sentence-transformers wikipedia



In [ ]:
# =========================================================
# Imports
# =========================================================

import os
from getpass import getpass

from langchain_community.document_loaders import WikipediaLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda, RunnablePassthrough

In [ ]:
# =========================================================
# Add Groq API key
# =========================================================

os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key: ")

In [ ]:
# =========================================================
# Load real documents from Wikipedia
# =========================================================
# This is the "real data" part.
# You can change the query to another topic.

loader = WikipediaLoader(
    query="Retrieval-augmented generation",
    load_max_docs=2
)

documents = loader.load()

print(f"Loaded {len(documents)} documents")

for i, doc in enumerate(documents):
    print("\n" + "-" * 70)
    print("Document:", i + 1)
    print("Source:", doc.metadata.get("source"))
    print("Title:", doc.metadata.get("title"))
    print(doc.page_content[:500])

In [ ]:
# =========================================================
# Split documents into chunks
# =========================================================

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=120
)

chunks = text_splitter.split_documents(documents)

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"chunk_{i}"

print(f"Number of chunks: {len(chunks)}")

print("\nExample chunk:")
print(chunks[0].page_content[:700])
print("\nMetadata:")
print(chunks[0].metadata)

In [ ]:
# =========================================================
# Create embeddings
# =========================================================

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

In [ ]:
# =========================================================
# Store chunks in Chroma vector database
# =========================================================

vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="real_wikipedia_rag_demo"
)

print("Chunks stored in Chroma.")

In [ ]:
# =========================================================
# Create retriever
# =========================================================

retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

question = "What is retrieval augmented generation?"

retrieved_docs = retriever.invoke(question)

print("Question:", question)

for doc in retrieved_docs:
    print("\n" + "-" * 70)
    print("Title:", doc.metadata.get("title"))
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Source:", doc.metadata.get("source"))
    print(doc.page_content[:600])

In [ ]:
# =========================================================
# Format retrieved chunks as context
# =========================================================

def format_docs(docs):
    formatted_docs = []

    for doc in docs:
        title = doc.metadata.get("title", "Unknown Title")
        source = doc.metadata.get("source", "Unknown Source")
        chunk_id = doc.metadata.get("chunk_id", "Unknown Chunk")

        formatted_docs.append(
            f"Title: {title}\n"
            f"Source: {source}\n"
            f"Chunk ID: {chunk_id}\n"
            f"Text: {doc.page_content}"
        )

    return "\n\n".join(formatted_docs)

In [ ]:
# =========================================================
# Create Groq LLM
# =========================================================

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [ ]:
# =========================================================
# Create RAG prompt
# =========================================================

prompt = ChatPromptTemplate.from_template("""
You are a careful RAG assistant.

Answer the question using only the context below.
If the answer is not in the context, say "I do not know."

Question:
{question}

Context:
{context}

Answer:
""")

In [ ]:
# =========================================================
# Build the LangChain RAG chain
# =========================================================

rag_chain = (
    {
        "context": retriever | RunnableLambda(format_docs),
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [ ]:
# =========================================================
# Ask a question
# =========================================================

question = "What is retrieval augmented generation?"

answer = rag_chain.invoke(question)

print("Question:")
print(question)

print("\nAnswer:")
print(answer)

In [ ]:
# =========================================================
# Ask more real questions
# =========================================================

questions = [
    "Why is retrieval augmented generation useful?",
    "What problem does RAG try to solve?",
    "How does RAG combine retrieval and generation?",
    "What are possible limitations of retrieval augmented generation?"
]

for q in questions:
    print("\n" + "=" * 90)
    print("Question:", q)
    print("\nAnswer:")
    print(rag_chain.invoke(q))

In [ ]:
# =========================================================
# Answer with sources
# =========================================================

def answer_with_sources(question):
    docs = retriever.invoke(question)
    context = format_docs(docs)

    answer = rag_chain.invoke(question)

    print("Question:")
    print(question)

    print("\nAnswer:")
    print(answer)

    print("\nRetrieved Sources:")
    for doc in docs:
        print("\n" + "-" * 70)
        print("Title:", doc.metadata.get("title"))
        print("Chunk ID:", doc.metadata.get("chunk_id"))
        print("Source:", doc.metadata.get("source"))
        print("Text Preview:")
        print(doc.page_content[:500])


answer_with_sources("Why is retrieval augmented generation useful?")